# Neural Network Backpropagation by Hand

## What this notebook is about

This notebook traces **one complete training step** of a tiny neural network by hand.
Every number is computed manually so you can see exactly what happens inside backpropagation.

Most frameworks (PyTorch, TensorFlow) do this automatically.
Understanding it manually builds the intuition for why training works.

## The three stages of one training step

| Stage | What happens |
|-------|--------------|
| **1. Forward pass** | Feed input through the network, get a prediction |
| **2. Backward pass** | Use the chain rule to compute how much each weight contributed to the error |
| **3. Weight update** | Nudge each weight in the direction that reduces the error |

## The Network Architecture

We use the smallest possible network that shows the full backprop chain:
- **2 inputs** (x1, x2)
- **1 hidden neuron** with sigmoid activation
- **1 output neuron** with sigmoid activation
- **3 weights** (w1, w2, w3) and **2 biases** (b1, b2)

```
  x1 --[w1]--\
               --> z1 = w1*x1 + w2*x2 + b1 --> a1 = sigmoid(z1) --[w3]--> z2 = w3*a1 + b2 --> a2 = sigmoid(z2) = y_hat
  x2 --[w2]--/
```

In full:
```
  z1   = w1*x1 + w2*x2 + b1      # hidden neuron linear step
  a1   = sigmoid(z1)              # hidden neuron activation
  z2   = w3*a1 + b2               # output neuron linear step
  a2   = sigmoid(z2)              # output = y_hat (prediction)
  Loss = 0.5 * (y - a2)^2        # how wrong we are
```

We use **MSE loss** with a 0.5 factor — this makes the derivative cleaner (the 2 cancels out).

## Step 1 — Define Inputs and Weights

In [ ]:
import numpy as np

# Inputs
x1, x2 = 1, 2
y = 1          # true label (what we want the network to predict)

# Initial weights and biases
w1, w2, w3 = 0.1, 0.2, 0.3
b1, b2     = 0.0, 0.0

lr = 0.1       # learning rate

sigmoid = lambda z: 1 / (1 + np.exp(-z))

print(f'Inputs:  x1={x1}, x2={x2},  true label y={y}')
print(f'Weights: w1={w1}, w2={w2}, w3={w3}')
print(f'Biases:  b1={b1}, b2={b2}')

## Step 2 — Forward Pass

We push the inputs through the network layer by layer.
At each neuron we do two things:
1. **Linear step** — compute a weighted sum: `z = w*x + b`
2. **Activation** — squash with sigmoid: `a = sigmoid(z)`

The sigmoid function squashes any number into the range (0, 1).
We use it here because our target y=1 is a probability.

In [ ]:
# Hidden layer
z1 = w1*x1 + w2*x2 + b1
a1 = sigmoid(z1)

# Output layer
z2 = w3*a1 + b2
a2 = sigmoid(z2)    # this is y_hat (our prediction)

print('--- Forward Pass ---')
print(f'z1 = w1*x1 + w2*x2 + b1 = {w1}*{x1} + {w2}*{x2} + {b1} = {z1}')
print(f'a1 = sigmoid({z1:.4f})    = {a1:.6f}')
print()
print(f'z2 = w3*a1 + b2          = {w3}*{a1:.6f} + {b2} = {z2:.6f}')
print(f'a2 = sigmoid({z2:.4f})    = {a2:.6f}   <-- prediction (y_hat)')

## Step 3 — Loss

We use **Mean Squared Error** with a 0.5 factor:

$$
Loss = \frac{1}{2}(y - \hat{y})^2
$$

The 0.5 is a convenience — when we differentiate, it cancels the exponent 2, giving us a cleaner gradient.

A smaller loss = prediction is closer to the true label.

In [ ]:
loss = 0.5 * (y - a2)**2

print(f'Loss = 0.5 * (y - a2)^2 = 0.5 * ({y} - {a2:.4f})^2 = {loss:.6f}')
print(f'\nOur prediction: {a2:.4f}  |  True label: {y}  |  Error: {y - a2:.4f}')

## Step 4 — Backward Pass (Chain Rule)

Now we work **backwards** through the network to find how much each weight
contributed to the loss. This is done using the **chain rule** from calculus.

The chain rule says: to find how the loss changes with respect to a weight deep in the network,
multiply the gradients of every step along the path from loss back to that weight.

```
Loss --> a2 --> z2 --> w3          (chain for w3)
Loss --> a2 --> z2 --> a1 --> z1 --> w2   (chain for w2)
Loss --> a2 --> z2 --> a1 --> z1 --> w1   (chain for w1)
```

**Two reusable pieces we compute once:**
```
dL/da2  = a2 - y                   (gradient of loss at output)
da2/dz2 = a2 * (1 - a2)           (sigmoid derivative at output)
```

**Sigmoid derivative:** If `a = sigmoid(z)`, then `da/dz = a * (1 - a)`.
This is why sigmoid is convenient — its derivative reuses the value we already computed.

In [ ]:
# --- Shared gradient pieces (computed once, reused) ---
dL_da2  = a2 - y           # gradient of loss w.r.t. prediction
da2_dz2 = a2 * (1 - a2)   # sigmoid derivative at output neuron

print('Shared gradient pieces:')
print(f'  dL/da2  = a2 - y        = {a2:.4f} - {y} = {dL_da2:.6f}')
print(f'  da2/dz2 = a2*(1-a2)     = {a2:.4f}*(1-{a2:.4f}) = {da2_dz2:.6f}')

### Gradient for w3

Path: Loss → a2 → z2 → w3

```
dL/dw3 = dL/da2 * da2/dz2 * dz2/dw3
       = (a2 - y) * a2*(1-a2) * a1
```

Note: `dz2/dw3 = a1` because `z2 = w3*a1 + b2`, so the derivative with respect to w3 is just a1.

In [ ]:
dz2_dw3 = a1   # because z2 = w3*a1 + b2

dL_dw3 = dL_da2 * da2_dz2 * dz2_dw3

print(f'dL/dw3 = dL/da2 * da2/dz2 * dz2/dw3')
print(f'       = {dL_da2:.4f} * {da2_dz2:.4f} * {dz2_dw3:.4f}')
print(f'       = {dL_dw3:.6f}')

### Gradients for w2 and w1

Path: Loss → a2 → z2 → a1 → z1 → w2 (or w1)

We need two more pieces to get back through the hidden layer:
```
dz2/da1 = w3              (because z2 = w3*a1 + b2)
da1/dz1 = a1*(1 - a1)    (sigmoid derivative at hidden neuron)
```

Then:
```
dL/dw2 = dL/da2 * da2/dz2 * dz2/da1 * da1/dz1 * dz1/dw2
       = (a2-y) * a2*(1-a2) * w3 * a1*(1-a1) * x2

dL/dw1 = same chain but multiply by x1 at the end
```

In [ ]:
dz2_da1 = w3                # because z2 = w3*a1 + b2
da1_dz1 = a1 * (1 - a1)    # sigmoid derivative at hidden neuron

dL_dw2 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * x2
dL_dw1 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * x1

print('Gradients for hidden layer weights:')
print(f'  dL/dw2 = {dL_dw2:.6f}')
print(f'  dL/dw1 = {dL_dw1:.6f}')
print()
print('Summary of all gradients:')
print(f'  dL/dw1 = {dL_dw1:.6f}   (w1 currently = {w1})')
print(f'  dL/dw2 = {dL_dw2:.6f}   (w2 currently = {w2})')
print(f'  dL/dw3 = {dL_dw3:.6f}   (w3 currently = {w3})')

## Step 5 — Weight Update

We update each weight by subtracting a small fraction of its gradient:

```
w_new = w_old - learning_rate * gradient
```

**Why subtract?**
The gradient points in the direction of **increasing** loss.
We want to go the other way (decreasing loss), so we subtract.

In [ ]:
w1_new = w1 - lr * dL_dw1
w2_new = w2 - lr * dL_dw2
w3_new = w3 - lr * dL_dw3

print(f'Learning rate: {lr}')
print()
print(f'w1: {w1} - {lr}*({dL_dw1:.6f}) = {w1_new:.6f}')
print(f'w2: {w2} - {lr}*({dL_dw2:.6f}) = {w2_new:.6f}')
print(f'w3: {w3} - {lr}*({dL_dw3:.6f}) = {w3_new:.6f}')

## Step 6 — Verify the Loss Decreased

We run one forward pass with the new weights to confirm the loss went down.
Even one small step should reduce the loss slightly.

In [ ]:
# Forward pass with updated weights
z1_new = w1_new*x1 + w2_new*x2 + b1
a1_new = sigmoid(z1_new)

z2_new = w3_new*a1_new + b2
a2_new = sigmoid(z2_new)

loss_new = 0.5 * (y - a2_new)**2

print(f'Prediction BEFORE update: {a2:.6f}   Loss: {loss:.6f}')
print(f'Prediction AFTER  update: {a2_new:.6f}   Loss: {loss_new:.6f}')
print()
if loss_new < loss:
    print(f'Loss reduced by {loss - loss_new:.8f} -- working correctly.')
else:
    print('Loss did not decrease -- try a smaller learning rate.')

## Step 7 — Full Training Loop

One step reduces the loss a tiny bit.
We repeat this hundreds of times and watch the loss shrink.

This is exactly what happens inside `model.fit()` in every ML framework.

In [ ]:
# Reset to initial weights
w1, w2, w3 = 0.1, 0.2, 0.3
b1, b2     = 0.0, 0.0
lr         = 0.1

loss_history = []

for step in range(500):
    # Forward
    z1 = w1*x1 + w2*x2 + b1
    a1 = sigmoid(z1)
    z2 = w3*a1 + b2
    a2 = sigmoid(z2)
    loss = 0.5 * (y - a2)**2
    loss_history.append(loss)

    # Backward
    dL_da2  = a2 - y
    da2_dz2 = a2 * (1 - a2)
    dz2_da1 = w3
    da1_dz1 = a1 * (1 - a1)

    dL_dw3 = dL_da2 * da2_dz2 * a1
    dL_dw2 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * x2
    dL_dw1 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * x1

    # Update
    w1 -= lr * dL_dw1
    w2 -= lr * dL_dw2
    w3 -= lr * dL_dw3

    if (step + 1) % 100 == 0:
        print(f'Step {step+1:4d}  |  Loss: {loss:.6f}  |  Prediction: {a2:.4f}')

print(f'\nFinal prediction: {a2:.6f}  (target was {y})')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.title('Loss Over 500 Training Steps')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()
print('Loss goes to near zero — the network learned to predict y=1.')

## Summary

| Step | What we did | Key formula |
|------|------------|-------------|
| Forward pass | Compute z1, a1, z2, a2 | z = w*x + b, a = sigmoid(z) |
| Loss | Measure how wrong we are | 0.5*(y - a2)^2 |
| Backward pass | Chain rule — trace gradient back through each layer | dL/dw = product of gradients along the path |
| Weight update | Move weights in the direction that reduces loss | w = w - lr * dL/dw |

## The Chain Rule in One Sentence

> To find how the loss changes with respect to a weight, multiply the gradient of every function
> along the path from the loss back to that weight.

## Why the Sigmoid Derivative Is Convenient

If `a = sigmoid(z)`, then `da/dz = a * (1 - a)`.
We already computed `a` in the forward pass, so the backward pass reuses it for free.

## What Libraries Do

PyTorch and TensorFlow do exactly these same steps automatically using **automatic differentiation**.
Understanding the manual version means you know what `.backward()` is actually computing.